
# Titanic — Machine Learning na prática

Este notebook foi feito para estudo prático de Machine Learning usando a base do Titanic.

## Objetivo
Prever a variável `Survived` a partir de atributos dos passageiros.

## Variáveis usadas
- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`

## Fluxo
1. Ler os dados
2. Tratar valores ausentes
3. Transformar variáveis categóricas
4. Separar treino e validação
5. Treinar o modelo
6. Avaliar resultado
7. Gerar predições para o `test.csv`


## Modelo deste notebook
**K-Nearest Neighbors (KNN)**

In [ ]:

import os
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if not os.path.exists("datas") and os.path.exists("../datas"):
    os.chdir("..")

train_df = pd.read_csv("datas/train.csv")
test_df = pd.read_csv("datas/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


In [ ]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

display(train_df[features + [target]].isnull().sum().to_frame('missing_values'))

fig = px.histogram(train_df, x='Sex', color='Survived', barmode='group',
                   title='Sobrevivência por sexo')
fig.show()

fig = px.histogram(train_df, x='Pclass', color='Survived', barmode='group',
                   title='Sobrevivência por classe')
fig.show()


## Pré-processamento

In [ ]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

X = train_df[features].copy()
y = train_df[target].copy()
X_test_final = test_df[features].copy()

# Tratamento de valores ausentes
for col in ['Age', 'Fare']:
    median_value = X[col].median()
    X[col] = X[col].fillna(median_value)
    X_test_final[col] = X_test_final[col].fillna(median_value)

mode_embarked = X['Embarked'].mode()[0]
X['Embarked'] = X['Embarked'].fillna(mode_embarked)
X_test_final['Embarked'] = X_test_final['Embarked'].fillna(mode_embarked)

# Codificação categórica
X = pd.get_dummies(X, columns=['Sex', 'Embarked'], drop_first=True)
X_test_final = pd.get_dummies(X_test_final, columns=['Sex', 'Embarked'], drop_first=True)

# Garantir mesmas colunas entre treino e teste
X, X_test_final = X.align(X_test_final, join='left', axis=1, fill_value=0)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
X_train.head()



## Escalonamento + KNN

O KNN compara distâncias entre pontos.
Por isso, ele funciona melhor quando as variáveis estão em escala semelhante.


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test_final)

model = KNeighborsClassifier(n_neighbors=7)
model.fit(X_train_scaled, y_train)


## Avaliação e geração da submissão

In [ ]:

valid_pred = model.predict(X_valid_scaled)
valid_acc = accuracy_score(y_valid, valid_pred)

print("Acurácia:", round(valid_acc, 4))
print("\nRelatório de classificação:")
print(classification_report(y_valid, valid_pred))

cm = confusion_matrix(y_valid, valid_pred)
cm_df = pd.DataFrame(cm, index=['Real_0', 'Real_1'], columns=['Pred_0', 'Pred_1'])
display(cm_df)

test_pred = model.predict(X_test_scaled)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('submission_knn.csv', index=False)
print("Arquivo salvo: submission_knn.csv")
submission.head()



## Teste extra com vários valores de k

Como o KNN depende do número de vizinhos, vale a pena testar diferentes valores.


In [ ]:

results = []
for k in range(3, 16, 2):
    temp_model = KNeighborsClassifier(n_neighbors=k)
    temp_model.fit(X_train_scaled, y_train)
    pred = temp_model.predict(X_valid_scaled)
    acc = accuracy_score(y_valid, pred)
    results.append({'k': k, 'accuracy': acc})

results_df = pd.DataFrame(results)
results_df
